# House Prices — Advanced Regression Techniques

**Goal:** Predict the sales price for each house based on 79 explanatory variables.

**Metric:** RMSE (Root Mean Squared Error on log-transformed prices).

**Kaggle link:** https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

## 1. Setup & Imports

In [ ]:
# Standard library
import warnings
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt

# Data manipulation
import numpy as np
import pandas as pd
import seaborn as sns

# Settings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

%matplotlib inline

## 2. Configuration

In [ ]:
# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../outputs/models")
SUBMISSIONS_DIR = Path("../outputs/submissions")

# Competition settings
COMPETITION_NAME = "house-prices-advanced-regression-techniques"
TARGET_COL = "SalePrice"
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 3. Load Data

In [ ]:
train_df = pd.read_csv(DATA_RAW / "train.csv")
test_df = pd.read_csv(DATA_RAW / "test.csv")

print(
    f"Train shape: {train_df.shape} — {train_df.shape[0]} houses, {train_df.shape[1]} columns (79 features + Id + SalePrice)"
)
print(
    f"Test shape:  {test_df.shape} — {test_df.shape[0]} houses, {test_df.shape[1]} columns (79 features + Id, no SalePrice)"
)

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train_df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"\nNumeric features: {len(numeric_cols) - 2} (excluding Id and SalePrice)")
print(f"Categorical features: {len(cat_cols)}")

## 4. Exploratory Data Analysis (EDA)

**Rules:** No modeling, no cleaning — only observation. Document every finding.

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

### 4.1 Missing Values

In [ ]:
# Missing values analysis for train and test
def missing_summary(df, name=""):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    result = pd.DataFrame({"count": missing, "percentage": missing_pct})
    result = result[result["count"] > 0].sort_values("percentage", ascending=False)
    print(f"=== {name} — {len(result)} columns with missing values ===")
    return result


train_missing = missing_summary(train_df, "Train")
train_missing

### 4.2 Target Distribution (SalePrice)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw distribution
train_df[TARGET_COL].hist(bins=50, ax=axes[0], edgecolor="black")
axes[0].set_title(f"{TARGET_COL} — Raw Distribution")
axes[0].set_xlabel("Price ($)")
axes[0].axvline(
    train_df[TARGET_COL].mean(), color="red", linestyle="--", label=f"Mean: ${train_df[TARGET_COL].mean():,.0f}"
)
axes[0].axvline(
    train_df[TARGET_COL].median(), color="green", linestyle="--", label=f"Median: ${train_df[TARGET_COL].median():,.0f}"
)
axes[0].legend()

# Log-transformed distribution (this is what Kaggle evaluates)
log_prices = np.log1p(train_df[TARGET_COL])
log_prices.hist(bins=50, ax=axes[1], edgecolor="black", color="orange")
axes[1].set_title(f"log(1 + {TARGET_COL}) — Closer to Normal")
axes[1].set_xlabel("Log Price")

# QQ plot of log-transformed prices
from scipy import stats

stats.probplot(log_prices, plot=axes[2])
axes[2].set_title("QQ Plot — log(1 + SalePrice)")

plt.tight_layout()
plt.show()

print(f"SalePrice range: ${train_df[TARGET_COL].min():,} — ${train_df[TARGET_COL].max():,}")
print(f"Mean: ${train_df[TARGET_COL].mean():,.0f} | Median: ${train_df[TARGET_COL].median():,.0f}")
print(f"Skewness (raw): {train_df[TARGET_COL].skew():.3f} → right-skewed, log transform helps")
print(f"Skewness (log): {log_prices.skew():.3f} → much closer to normal")

### 4.3 Correlations with SalePrice

In [ ]:
# Top 15 features most correlated with SalePrice
corr_with_target = (
    train_df.select_dtypes(include=[np.number]).corr()[TARGET_COL].drop([TARGET_COL, "Id"]).sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
top_15 = corr_with_target.head(15)
sns.barplot(x=top_15.values, y=top_15.index, palette="viridis", ax=ax)
ax.set_title("Top 15 Numeric Features — Correlation with SalePrice")
ax.set_xlabel("Pearson Correlation")
for i, v in enumerate(top_15.values):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

print("Key insight: OverallQual (0.79) and GrLivArea (0.71) are the strongest predictors.")
print("Garage, basement, and year features also matter significantly.")

In [ ]:
# Correlation heatmap — top 10 features + SalePrice
top_features = corr_with_target.head(10).index.tolist() + [TARGET_COL]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    train_df[top_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5,
)
ax.set_title("Correlation Heatmap — Top 10 Features + SalePrice")
plt.tight_layout()
plt.show()

# Watch for multicollinearity
print("Multicollinearity to watch:")
print("  - GarageCars ↔ GarageArea (0.88) — keep one")
print("  - TotalBsmtSF ↔ 1stFlrSF (0.82) — related")
print("  - GrLivArea ↔ TotRmsAbvGrd (0.83) — related")
print("  - YearBuilt ↔ GarageYrBlt (0.83) — garage built same year as house")

### 4.4 Key Feature Relationships with SalePrice

In [ ]:
# Scatter plots — top 4 numeric features vs SalePrice
top_4 = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, col in enumerate(top_4):
    axes[i].scatter(train_df[col], train_df[TARGET_COL], alpha=0.3, s=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(TARGET_COL if i == 0 else "")
    axes[i].set_title(f"{col} vs {TARGET_COL}")

plt.tight_layout()
plt.show()

# GrLivArea outliers: two houses >4000 sqft with low price — likely partial sales
outliers = train_df[(train_df["GrLivArea"] > 4000) & (train_df[TARGET_COL] < 300000)]
print(f"Potential outliers: {len(outliers)} houses with GrLivArea > 4000 sqft but SalePrice < $300k")
print(outliers[["Id", "GrLivArea", TARGET_COL, "SaleCondition"]])

### 4.5 Categorical Features

In [ ]:
# Categorical features — cardinality overview
cat_info = pd.DataFrame(
    {
        "nunique": train_df[cat_cols].nunique(),
        "missing_pct": (train_df[cat_cols].isnull().sum() / len(train_df) * 100).round(1),
    }
).sort_values("nunique", ascending=False)
print("Categorical features — cardinality and missing %:")
cat_info

In [ ]:
# Top categorical features — median SalePrice by category
key_cats = ["Neighborhood", "ExterQual", "KitchenQual", "BsmtQual", "GarageFinish"]
fig, axes = plt.subplots(1, len(key_cats), figsize=(24, 6))

for i, col in enumerate(key_cats):
    order = train_df.groupby(col)[TARGET_COL].median().sort_values(ascending=False).index
    sns.boxplot(data=train_df, x=col, y=TARGET_COL, order=order, ax=axes[i])
    axes[i].set_title(f"{TARGET_COL} by {col}")
    axes[i].tick_params(axis="x", rotation=45)
    if i > 0:
        axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

print("Key insight: Quality features (ExterQual, KitchenQual, BsmtQual) show clear price ordering.")
print("Neighborhood is the strongest categorical predictor — large price variation between neighborhoods.")

### 4.6 EDA Summary

**Dataset:** 1460 train / 1459 test houses, 79 features (36 numeric + 43 categorical).

**Target (SalePrice):**
- Range: $34,900 — $755,000 (median $163,000)
- Right-skewed (1.88) → log transform makes it near-normal (0.12)
- Kaggle evaluates RMSE on log-prices, so we should train on log(SalePrice)

**Missing values:**
- 19 columns have missing values in train
- PoolQC (99.5%), MiscFeature (96.3%), Alley (93.8%), Fence (80.8%): NA means "absent"
- MasVnrType (59.7%): NA likely means no masonry veneer
- Garage* (5.5%) and Bsmt* (2.5%): NA means no garage / no basement
- LotFrontage (17.7%): genuinely missing, impute by neighborhood median
- Electrical (1 row): impute with mode

**Top numeric predictors:** OverallQual (0.79), GrLivArea (0.71), GarageCars (0.64), GarageArea (0.62), TotalBsmtSF (0.61)

**Multicollinearity:** GarageCars↔GarageArea, TotalBsmtSF↔1stFlrSF, GrLivArea↔TotRmsAbvGrd — may need to drop one of each pair

**Top categorical predictors:** Neighborhood, ExterQual, KitchenQual, BsmtQual, GarageFinish

**Outliers:** 2 houses with GrLivArea > 4000 sqft but SalePrice < $300k — consider removing

**Next steps (Data Cleaning):**
1. Handle "absent" NAs (fill with "None" for categorical, 0 for numeric)
2. Impute LotFrontage by neighborhood median
3. Impute remaining missing values (mode for categorical, median for numeric)
4. Consider removing the 2 GrLivArea outliers

## 5. Data Cleaning

In [ ]:
# Remove 2 GrLivArea outliers identified in EDA (>4000 sqft, <$300k)
# WHY: These are likely partial sales that would distort the model
outlier_idx = train_df[(train_df["GrLivArea"] > 4000) & (train_df[TARGET_COL] < 300000)].index
print(f"Removing {len(outlier_idx)} outliers (Ids: {train_df.loc[outlier_idx, 'Id'].tolist()})")
train_df = train_df.drop(outlier_idx).reset_index(drop=True)
print(f"Train shape after outlier removal: {train_df.shape}")

### 5.1 Handle "Feature Absent" NAs

Per `data_description.txt`, NA in many columns means the feature doesn't exist (no pool, no garage, no basement, etc.). These are not truly missing — fill with `"None"` (categorical) or `0` (numeric).

In [ ]:
# Columns where NA means "feature absent" — fill with "None" or 0
# WHY: data_description.txt documents NA as a valid category (e.g., "NA = No Pool")

absent_cat_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType",
]
absent_num_cols = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "MasVnrArea",
]

for df in [train_df, test_df]:
    for col in absent_cat_cols:
        df[col] = df[col].fillna("None")
    for col in absent_num_cols:
        df[col] = df[col].fillna(0)

print("Filled 'feature absent' NAs:")
print(f"  Categorical ({len(absent_cat_cols)} cols): → 'None'")
print(f"  Numeric ({len(absent_num_cols)} cols): → 0")

### 5.2 Impute LotFrontage by Neighborhood Median

WHY: Lots in the same neighborhood tend to have similar frontage. Neighborhood median is a better imputation than global median.

In [ ]:
# Impute LotFrontage — fit on train, apply to both
# WHY: No data leakage — stats computed on train only
lot_frontage_by_neighborhood = train_df.groupby("Neighborhood")["LotFrontage"].median()

for df in [train_df, test_df]:
    for idx in df[df["LotFrontage"].isnull()].index:
        neighborhood = df.loc[idx, "Neighborhood"]
        median_val = lot_frontage_by_neighborhood.get(neighborhood, train_df["LotFrontage"].median())
        df.loc[idx, "LotFrontage"] = median_val

print("LotFrontage imputed by neighborhood median (fitted on train)")
print(f"  Train missing: {train_df['LotFrontage'].isnull().sum()}")
print(f"  Test missing:  {test_df['LotFrontage'].isnull().sum()}")

### 5.3 Impute Remaining Missing Values

Strategy: mode for categorical, median for numeric. All stats fitted on train only.

In [ ]:
# Impute remaining missing values — fit on train, apply to both
# WHY: Some columns have a few remaining NAs (Electrical in train, MSZoning/Functional/etc. in test)

# Compute fill values from train
cat_fill = {}
num_fill = {}

for col in train_df.columns:
    if col in ["Id", TARGET_COL]:
        continue
    if train_df[col].dtype.kind in ("f", "i", "u"):  # float, int, unsigned int
        num_fill[col] = train_df[col].median()
    else:
        cat_fill[col] = train_df[col].mode().iloc[0] if not train_df[col].mode().empty else "None"

# Apply to both datasets
for df in [train_df, test_df]:
    for col, val in cat_fill.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    for col, val in num_fill.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

print("Remaining NAs imputed (mode for categorical, median for numeric)")
print(f"  Train total missing: {train_df.drop(columns=['Id', TARGET_COL]).isnull().sum().sum()}")
print(f"  Test total missing:  {test_df.drop(columns=['Id']).isnull().sum().sum()}")

### 5.4 Verification

In [ ]:
# Final verification — zero missing values in both datasets
train_missing_final = train_df.drop(columns=["Id", TARGET_COL]).isnull().sum().sum()
test_missing_final = test_df.drop(columns=["Id"]).isnull().sum().sum()

print(f"Train missing values: {train_missing_final}")
print(f"Test missing values:  {test_missing_final}")
assert train_missing_final == 0, "Train still has missing values!"
assert test_missing_final == 0, "Test still has missing values!"
print("\n✓ No missing values remain in train or test.")

print("\nFinal shapes:")
print(f"  Train: {train_df.shape}")
print(f"  Test:  {test_df.shape}")

## 6. Feature Engineering

In [ ]:
# Add src/ to path for imports
import sys

sys.path.insert(0, str(Path("../").resolve()))

from src.features import apply_feature_engineering

# Save test IDs before transformation (needed for submission)
test_ids = test_df["Id"].copy()

# Apply full feature engineering pipeline
train_df, test_df = apply_feature_engineering(train_df, test_df, target_col=TARGET_COL)

print("After feature engineering:")
print(f"  Train: {train_df.shape}")
print(f"  Test:  {test_df.shape}")
print("  New features: TotalSF, TotalBath, TotalPorchSF, HouseAge, RemodAge, IsRemodeled, Has*")
print("  Ordinal encoded: quality/condition features")
print("  One-hot encoded: remaining categoricals")
print("  Dropped: Id, Utilities, Street, PoolQC, MiscFeature, MiscVal, Alley")
print(f"\nNo object columns remain: {len(train_df.select_dtypes(include=['object', 'string']).columns) == 0}")

In [ ]:
# Verify: no missing values after feature engineering
assert train_df.drop(columns=[TARGET_COL]).isnull().sum().sum() == 0, "Train has NaN after feature engineering!"
assert test_df.isnull().sum().sum() == 0, "Test has NaN after feature engineering!"
print("✓ No missing values after feature engineering")

# Verify: no object columns remain (all encoded)
train_obj = train_df.select_dtypes(include=["object", "string"]).columns.tolist()
test_obj = test_df.select_dtypes(include=["object", "string"]).columns.tolist()
assert len(train_obj) == 0, f"Train still has object columns: {train_obj}"
assert len(test_obj) == 0, f"Test still has object columns: {test_obj}"
print("✓ All categorical features encoded")

# Verify: train and test have same columns (minus target)
train_feat_cols = set(train_df.columns) - {TARGET_COL}
test_feat_cols = set(test_df.columns)
assert train_feat_cols == test_feat_cols, f"Column mismatch: {train_feat_cols.symmetric_difference(test_feat_cols)}"
print(f"✓ Train and test aligned: {len(train_feat_cols)} features each")

## 7. Modeling

**Baseline approach:** RandomForestRegressor with default hyperparameters (no tuning — YAGNI).
Train on `log(1 + SalePrice)` since Kaggle evaluates RMSLE. K-Fold cross-validation with k=5.

In [ ]:
# Prepare features and target
# WHY: Train on log(SalePrice) — Kaggle metric is RMSE on log-transformed prices
X = train_df.drop(columns=[TARGET_COL])
y = np.log1p(train_df[TARGET_COL])  # log(1 + price)

print(f"Features: {X.shape}")
print(f"Target: log(1 + SalePrice), range [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
# Baseline model: RandomForestRegressor with default hyperparameters
# WHY: Works well out-of-the-box, no scaling needed, provides feature importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

# Cross-validation: 5-fold, scoring = neg_root_mean_squared_error
# WHY: RMSE on log-prices matches Kaggle's evaluation metric
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring="neg_root_mean_squared_error")

# Convert negative scores to positive RMSE
cv_rmse = -cv_scores

print("=== Baseline Model: RandomForestRegressor ===")
print(f"CV RMSLE scores: {cv_rmse.round(4)}")
print(f"CV RMSLE mean:   {cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")
print("\nThis is our baseline to beat in future iterations.")

## 8. Evaluation

In [ ]:
# Train on full training set for evaluation plots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

# Metrics on validation set
val_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
val_mae = mean_absolute_error(y_val, y_pred)

print(f"Validation RMSLE: {val_rmse:.4f}")
print(f"Validation MAE (log-scale): {val_mae:.4f}")

# Prediction vs Actual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predicted vs actual (log scale)
axes[0].scatter(y_val, y_pred, alpha=0.3, s=10)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--", lw=2)
axes[0].set_xlabel("Actual log(1 + SalePrice)")
axes[0].set_ylabel("Predicted log(1 + SalePrice)")
axes[0].set_title(f"Predicted vs Actual (RMSLE: {val_rmse:.4f})")

# Residuals distribution
residuals = y_val - y_pred
axes[1].hist(residuals, bins=40, edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual (actual - predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residuals Distribution")

plt.tight_layout()
plt.show()

print(f"\nResiduals: mean={residuals.mean():.4f}, std={residuals.std():.4f}")

In [ ]:
# Feature importance — top 20
importance_df = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": model.feature_importances_,
    }
).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=importance_df.head(20), x="importance", y="feature", palette="viridis", ax=ax)
ax.set_title("Top 20 Feature Importance (RandomForest)")
plt.tight_layout()
plt.show()

print("Top 5 features:")
for _, row in importance_df.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

## 9. Submission

Retrain on 100% of training data (not 80%), predict test set, validate format, save CSV.

In [ ]:
# Retrain on 100% of training data
# WHY: For submission, use all available training data to maximize model performance
model_final = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
model_final.fit(X, y)
print(f"Model trained on {X.shape[0]} samples, {X.shape[1]} features")

# Generate predictions on test set
# WHY: model predicts log(1+price), expm1 converts back to actual prices
test_predictions_log = model_final.predict(test_df)
test_predictions = np.expm1(test_predictions_log)  # inverse of log1p

print(f"Predictions range: ${test_predictions.min():,.0f} — ${test_predictions.max():,.0f}")
print(f"Predictions mean: ${test_predictions.mean():,.0f}")

In [ ]:
# Create and validate submission file
from src.utils import save_submission

submission = save_submission(
    predictions=pd.Series(test_predictions),
    ids=test_ids,
    target_col=TARGET_COL,
    output_path=SUBMISSIONS_DIR / "submission_001_rf_baseline.csv",
)

# Validate against sample submission
sample_sub = pd.read_csv(DATA_RAW / "sample_submission.csv")
assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == list(sample_sub.columns), f"Column mismatch: {list(submission.columns)}"
assert (submission["Id"] == sample_sub["Id"]).all(), "Id mismatch"
assert submission[TARGET_COL].notna().all(), "Predictions contain NaN"
assert (submission[TARGET_COL] > 0).all(), "Predictions contain non-positive values"

print("✓ Submission validated against sample_submission.csv")
print(f"  Shape: {submission.shape}")
print(f"  Saved to: {SUBMISSIONS_DIR / 'submission_001_rf_baseline.csv'}")
submission.head(10)

## 10. Iteration 1 — Model Comparison (XGBoost / LightGBM)

**Goal:** Replace RandomForest with gradient boosting models. Same features, only the model changes.

In [ ]:
# Model comparison: RandomForest vs XGBoost vs LightGBM
# WHY: Gradient boosting typically outperforms RF on tabular data
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models_comparison = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

results = {}
print("=== Model Comparison (5-fold CV RMSLE) ===")
print(f"{'Model':<15} {'Mean':>8} {'Std':>8}  {'vs Baseline':>12}")
print("-" * 55)

for name, mdl in models_comparison.items():
    scores = -cross_val_score(mdl, X, y, cv=kf, scoring="neg_root_mean_squared_error")
    results[name] = {"mean": scores.mean(), "std": scores.std(), "scores": scores}
    baseline_diff = scores.mean() - results["RandomForest"]["mean"] if "RandomForest" in results else 0
    sign = "+" if baseline_diff >= 0 else ""
    print(f"{name:<15} {scores.mean():>8.4f} {scores.std():>8.4f}  {sign}{baseline_diff:>11.4f}")

best_model_name = min(results, key=lambda k: results[k]["mean"])
print(f"\n→ Best model: {best_model_name} (CV RMSLE: {results[best_model_name]['mean']:.4f})")

In [ ]:
# Evaluation of LightGBM on validation set
best_model = LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
best_model.fit(X_train, y_train)
y_pred_lgbm = best_model.predict(X_val)

val_rmse_lgbm = np.sqrt(mean_squared_error(y_val, y_pred_lgbm))
val_mae_lgbm = mean_absolute_error(y_val, y_pred_lgbm)

print(f"LightGBM Validation RMSLE: {val_rmse_lgbm:.4f} (RF was {val_rmse:.4f})")
print(f"LightGBM Validation MAE:   {val_mae_lgbm:.4f} (RF was {val_mae:.4f})")

# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_val, y_pred_lgbm, alpha=0.3, s=10)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--", lw=2)
axes[0].set_xlabel("Actual log(1 + SalePrice)")
axes[0].set_ylabel("Predicted log(1 + SalePrice)")
axes[0].set_title(f"LightGBM — Predicted vs Actual (RMSLE: {val_rmse_lgbm:.4f})")

residuals_lgbm = y_val - y_pred_lgbm
axes[1].hist(residuals_lgbm, bins=40, edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual (actual - predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title("LightGBM — Residuals Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Generate submission with LightGBM (retrained on 100% data)
model_lgbm_final = LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
model_lgbm_final.fit(X, y)

test_pred_lgbm_log = model_lgbm_final.predict(test_df)
test_pred_lgbm = np.expm1(test_pred_lgbm_log)

submission_lgbm = save_submission(
    predictions=pd.Series(test_pred_lgbm),
    ids=test_ids,
    target_col=TARGET_COL,
    output_path=SUBMISSIONS_DIR / "submission_002_lgbm_default.csv",
)

# Validate
assert submission_lgbm.shape == sample_sub.shape
assert (submission_lgbm["Id"] == sample_sub["Id"]).all()
assert submission_lgbm[TARGET_COL].notna().all()
assert (submission_lgbm[TARGET_COL] > 0).all()

print("✓ LightGBM submission validated")
print(f"  Predictions range: ${test_pred_lgbm.min():,.0f} — ${test_pred_lgbm.max():,.0f}")
print(f"  Saved to: {SUBMISSIONS_DIR / 'submission_002_lgbm_default.csv'}")
submission_lgbm.head()

## 11. Iteration 2 — Hyperparameter Tuning (LightGBM)

**Goal:** Optimize LightGBM params with RandomizedSearchCV. Same features, only hyperparameters change.

In [ ]:
# Hyperparameter tuning with RandomizedSearchCV
# WHY: Default params are a good starting point, but tuning can squeeze extra performance
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "n_estimators": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "max_depth": [3, 5, 7, 9, -1],
    "num_leaves": [15, 31, 50, 70, 100],
    "min_child_samples": [5, 10, 20, 30, 50],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 0.5, 1.0],
    "reg_lambda": [0, 0.01, 0.1, 0.5, 1.0],
}

search = RandomizedSearchCV(
    LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    param_distributions,
    n_iter=50,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

search.fit(X, y)

print("=== Hyperparameter Tuning Results ===")
print(f"Best CV RMSLE:    {-search.best_score_:.4f}")
print("Default LightGBM: 0.1266")
print(f"Improvement:      {0.1266 - (-search.best_score_):.4f}")
print("\nBest parameters:")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

In [ ]:
# Evaluation of tuned LightGBM
tuned_model = search.best_estimator_
tuned_model.fit(X_train, y_train)
y_pred_tuned = tuned_model.predict(X_val)

val_rmse_tuned = np.sqrt(mean_squared_error(y_val, y_pred_tuned))
print(f"Tuned LightGBM Validation RMSLE: {val_rmse_tuned:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_val, y_pred_tuned, alpha=0.3, s=10)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--", lw=2)
axes[0].set_xlabel("Actual log(1 + SalePrice)")
axes[0].set_ylabel("Predicted log(1 + SalePrice)")
axes[0].set_title(f"Tuned LightGBM — Predicted vs Actual (RMSLE: {val_rmse_tuned:.4f})")

residuals_tuned = y_val - y_pred_tuned
axes[1].hist(residuals_tuned, bins=40, edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual (actual - predicted)")
axes[1].set_ylabel("Count")
axes[1].set_title("Tuned LightGBM — Residuals Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Generate submission with tuned LightGBM (retrained on 100% data)
model_tuned_final = LGBMRegressor(**search.best_params_, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
model_tuned_final.fit(X, y)

test_pred_tuned_log = model_tuned_final.predict(test_df)
test_pred_tuned = np.expm1(test_pred_tuned_log)

submission_tuned = save_submission(
    predictions=pd.Series(test_pred_tuned),
    ids=test_ids,
    target_col=TARGET_COL,
    output_path=SUBMISSIONS_DIR / "submission_003_lgbm_tuned.csv",
)

# Validate
assert submission_tuned.shape == sample_sub.shape
assert (submission_tuned["Id"] == sample_sub["Id"]).all()
assert submission_tuned[TARGET_COL].notna().all()
assert (submission_tuned[TARGET_COL] > 0).all()

print("✓ Tuned LightGBM submission validated")
print(f"  Predictions range: ${test_pred_tuned.min():,.0f} — ${test_pred_tuned.max():,.0f}")
print(f"  Saved to: {SUBMISSIONS_DIR / 'submission_003_lgbm_tuned.csv'}")
submission_tuned.head()